# NetZeroAI — Exploratory Data Analysis

Run from the project root (`jupyter notebook notebook/EDA.ipynb`) so relative paths resolve. Covers: why the originally attached `global_warming_dataset.csv` was rejected, and the real OWID CO2/Energy data used instead.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()))

import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()

## 1. Why the synthetic dataset was rejected

`global_warming_dataset.csv` is 100k rows of `Country_1`..`Country_195` synthetic labels. Row-level and country-aggregated correlations are both at chance level (max |r| ~0.01-0.21, the latter fully explained by 1/sqrt(n) noise averaging over ~513 rows/country).

In [ ]:
synthetic = pd.read_csv(ROOT / "global_warming_dataset.csv")
num_cols = synthetic.select_dtypes(include="number").columns.drop("Year")
corr = synthetic[num_cols].corr()
np.fill_diagonal(corr.values, np.nan)
print("Max |correlation| (row-level):", np.nanmax(np.abs(corr.values)))

## 2. Real data: OWID CO2 + Energy, merged at the 2022 reference year

See `src/components/data_ingestion.py` for the full filtering/merge/imputation logic. This just inspects the result.

In [ ]:
from src.components.data_ingestion import DataIngestion

train_path, test_path = DataIngestion().initiate_data_ingestion()
df = pd.read_csv(train_path)
print(df.shape)
df.describe()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

indicator_cols = ["co2_per_capita", "ghg_per_capita", "co2pc_trend_slope", "renewables_share_elec", "gdp_per_capita"]
sns.pairplot(df[indicator_cols].dropna())
plt.show()